# Thanh khoản, khớp lệnh và chi phí

Vị thế được khóa tại `t`, vào tại giá đóng cửa `t+1` nếu có giá hợp lệ và thoát dự kiến ở `t+2`. Phần vào chưa khớp để tiền mặt. Nếu thiếu giá thoát, vị thế chiếm hạn mức và chờ giá hợp lệ tiếp theo. [Notebook chính](../jpx_overview.ipynb)

In [1]:
from pathlib import Path
import json

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "requirements.txt").is_file()
            and (p / "src/__init__.py").is_file())
results = root / "artifacts"
if not (results / "development_summary.csv").is_file():
    raise FileNotFoundError("Tạo các bảng kết quả theo reports/reproduce.md trước khi chạy notebook.")
plt.rcParams.update({"figure.dpi": 115, "axes.spines.top": False, "axes.spines.right": False})
blue, green, red, gray = "#285A82", "#24796C", "#A63D40", "#687783"


## Độ phủ trên tập kiểm thử

In [2]:
test = pd.read_csv(results / "historical_summary.csv").set_index("Model")
strict = pd.read_csv(results / "backtest_daily.csv").groupby("Model")["Gross"].count()
execution = pd.read_csv(results / "execution_coverage.csv").set_index("Model")
coverage = test.loc[execution.index, ["Sessions", "Scored sessions"]].assign(**{"Phiên đủ giá toàn rổ": strict})
display(coverage.rename(columns={"Sessions": "Phiên tín hiệu có t+2", "Scored sessions": "Phiên chấm riêng"})
        .rename_axis("Mô hình"))
execution["Max deferred notional"] = execution["Max deferred notional"].mul(100)
display(execution[["Orders", "Filled", "Unfilled", "Blocked", "Deferred", "Unresolved", "Max deferred notional"]]
        .rename(columns={"Orders": "Dự kiến", "Filled": "Đã vào", "Unfilled": "Thiếu giá vào",
            "Blocked": "Bị chặn", "Deferred": "Quá hạn thoát", "Unresolved": "Còn mở cuối kỳ",
            "Max deferred notional": "Tỷ trọng quá hạn cao nhất (%)",
        }).rename_axis("Mô hình").round(2))

,Phiên tín hiệu có t+2,Phiên chấm riêng,Phiên đủ giá toàn rổ
Mô hình,,,
Ridge six,130,125,2
LightGBM finance,130,129,51


,Dự kiến,Đã vào,Thiếu giá vào,Bị chặn,Quá hạn thoát,Còn mở cuối kỳ,Tỷ trọng quá hạn cao nhất (%)
Mô hình,,,,,,,
Ridge six,52000,51671,114,215,264,4,5.69
LightGBM finance,52000,51935,41,24,49,1,2.00


Một danh mục có 400 mã. Chỉ một vị thế thiếu giá có thể khiến phép chấm đủ giá loại cả phiên. Bảng khớp lệnh đếm riêng lệnh thiếu giá vào, lệnh bị chặn và vị thế đã vào nhưng quá hạn thoát. Vị thế còn mở cuối kỳ là một phần của số vị thế quá hạn. [Định nghĩa từng trạng thái](../jpx_overview.ipynb#Mô-phỏng-vào-và-thoát-vị-thế-trên-tập-kiểm-thử)

## Ba cách nhìn về chi phí

In [3]:
strict_cost = pd.read_csv(results / "backtest_costs.csv").query("Scope == 'Common' and `Borrow bps/year` == 0")
proxy = pd.read_csv(results / "target_weight_proxy.csv")
executed = pd.read_csv(results / "execution_costs.csv").query("`Borrow bps/year` == 0")
display(strict_cost[["Model", "Fee bps", "Net bps", "Priced sessions"]].rename(columns={
    "Model": "Mô hình", "Fee bps": "Phí một chiều (bps)", "Net bps": "Lãi ròng (bps/phiên)",
    "Priced sessions": "Phiên đủ giá chung",
}).round(2))
display(proxy[["Model", "Cost rate bps", "Net bps", "Common sessions"]].rename(columns={
    "Model": "Mô hình", "Cost rate bps": "Phí một chiều (bps)", "Net bps": "Lãi ròng ước tính (bps/phiên)",
    "Common sessions": "Phiên đủ nhãn chung",
}).round(2))
display(executed[["Model", "Fee bps", "Recognized bps per session", "Execution sessions"]].rename(columns={
    "Model": "Mô hình", "Fee bps": "Phí một chiều (bps)",
    "Recognized bps per session": "Lãi ròng ghi nhận (bps/phiên)", "Execution sessions": "Phiên thực thi",
}).round(2))


,Mô hình,Phí một chiều (bps),Lãi ròng (bps/phiên),Phiên đủ giá chung
12,LightGBM finance,0,56.17,1
15,LightGBM finance,2,52.64,1
18,LightGBM finance,5,47.34,1
21,LightGBM finance,10,38.50,1
36,Ridge six,0,-56.06,1
39,Ridge six,2,-64.00,1
42,Ridge six,5,-75.92,1
45,Ridge six,10,-95.79,1


,Mô hình,Phí một chiều (bps),Lãi ròng ước tính (bps/phiên),Phiên đủ nhãn chung
0,Ridge six,0,-14.05,125
1,Ridge six,2,-16.36,125
2,Ridge six,5,-19.83,125
3,Ridge six,10,-25.61,125
4,LightGBM finance,0,2.15,125
5,LightGBM finance,2,-1.16,125
6,LightGBM finance,5,-6.14,125
7,LightGBM finance,10,-14.44,125


,Mô hình,Phí một chiều (bps),Lãi ròng ghi nhận (bps/phiên),Phiên thực thi
0,Ridge six,0,-12.78,131
3,Ridge six,2,-15.03,131
6,Ridge six,5,-18.41,131
9,Ridge six,10,-24.04,131
12,LightGBM finance,0,2.72,131
15,LightGBM finance,2,-0.57,131
18,LightGBM finance,5,-5.50,131
21,LightGBM finance,10,-13.73,131


Bảng đầu chỉ dùng các phiên đủ mọi giá. Bảng thứ hai dùng nhãn cùng thay đổi tỷ trọng mục tiêu. Bảng cuối tính giao dịch đã mô phỏng và lợi nhuận ghi nhận khi thoát. Mẫu số, thời điểm ghi nhận và rủi ro còn mở của ba bảng khác nhau. Giả định khớp tại giá đóng cửa hiện chưa phản ánh trượt giá, phí vay thực tế hay cổ tức.

## Kiểm tra trên các cửa sổ phát triển

In [4]:
liquidity = pd.read_csv(results / "development_liquidity.csv")
groups = liquidity.loc[liquidity["Book"].ne("Toàn tập") & liquidity["Leg size"].gt(0)]
spread = groups.pivot(index="Date", columns=["Model", "Book"], values="Gross bps").dropna()
display(spread.mean().rename("Spread (bps/phiên)").to_frame().assign(**{"Phiên chung": len(spread)}).rename_axis(index=["Mô hình", "Nhóm thanh khoản"]).round(2))
orders = pd.read_csv(results / "development_execution_orders.csv", parse_dates=["Date", "Planned exit", "Exit"])
orders["Vế"] = orders["Weight"].gt(0).map({True: "Mua", False: "Bán khống"})
orders["Quá hạn thoát"] = orders["Exit"].gt(orders["Planned exit"]) | orders["Status"].eq("open")
orders["Thiếu giá vào"] = orders["Status"].eq("unfilled")
orders["Tỷ lệ danh nghĩa đã vào"] = orders["Filled weight"].abs().div(orders["Weight"].abs())
rates = orders.groupby(["Model", "Vế"])[["Thiếu giá vào", "Quá hạn thoát", "Tỷ lệ danh nghĩa đã vào"]].mean()
display(rates.mul(100).add_suffix(" (%)").rename_axis(index=["Mô hình", "Vế"]).round(2))
liquidity_pct = orders.groupby(["Model", "Date"])["Liquidity"].rank(pct=True)
orders["Thanh khoản trong các mã được chọn"] = pd.cut(
    liquidity_pct, [0, 1 / 3, 2 / 3, 1], labels=["Thấp", "Vừa", "Cao"], include_lowest=True,
)
by_liquidity = orders.groupby(["Model", "Thanh khoản trong các mã được chọn"], observed=True)
display(by_liquidity.agg(**{
    "Thiếu giá vào (%)": ("Thiếu giá vào", lambda values: values.mean() * 100),
    "Quá hạn thoát (%)": ("Quá hạn thoát", lambda values: values.mean() * 100),
    "Số lệnh dự kiến": ("Thiếu giá vào", "size"),
}).rename_axis(index=["Mô hình", "Nhóm thanh khoản"]).round(2))
misses = orders.loc[orders["Thiếu giá vào"] | orders["Quá hạn thoát"]]
display(misses.groupby("SecuritiesCode").size().nlargest(8).rename("Số lệnh gặp thiếu giá").rename_axis("Mã cổ phiếu"))
display(misses.groupby(misses["Date"].dt.to_period("M")).size().nlargest(6).rename("Số lệnh gặp thiếu giá theo tháng").rename_axis("Tháng"))
sensitivity = pd.read_csv(results / "development_sensitivity.csv")
main = pd.read_csv(results / "development_daily.csv", usecols=["Date", "Model", "Spread"])
main = main.loc[main["Model"].isin(sensitivity["Model"])].assign(**{"Portfolio size": 200, "Weight ratio": 2.})
cases = pd.concat([sensitivity, main], ignore_index=True).pivot(
    index="Date", columns=["Model", "Portfolio size", "Weight ratio"], values="Spread",
)
shared = cases.dropna()
display(pd.DataFrame({"Phiên riêng": cases.count(), "Phiên chung": len(shared),
                      "Spread (bps/phiên)": shared.mean().mul(10_000),
                      "Sharpe": shared.mean().div(shared.std())})
        .rename_axis(index=["Mô hình", "Số mã mỗi vế", "Tỷ lệ trọng số cực trị"]).round(4))


Spread (bps/phiên)  Phiên chung
Mô hình          Nhóm thanh khoản                                 
Ridge six        Thấp                           11.55          680
                 Vừa                            13.25          680
                 Cao                             9.75          680
LightGBM finance Thấp                           12.49          680
                 Vừa                             6.36          680
                 Cao                            -1.55          680

Thiếu giá vào (%)  Quá hạn thoát (%)  \
Mô hình          Vế                                                
LightGBM finance Bán khống               0.04               0.21   
                 Mua                     0.03               0.20   
Ridge six        Bán khống               0.15               0.59   
                 Mua                     0.29               0.36   

                            Tỷ lệ danh nghĩa đã vào (%)  
Mô hình          Vế                                      
LightGBM finance Bán khống                        99.68  
                 Mua                              99.73  
Ridge six        Bán khống                        98.49  
                 Mua                              99.14

Thiếu giá vào (%)  Quá hạn thoát (%)  \
Mô hình          Nhóm thanh khoản                                         
LightGBM finance Thấp                           0.01               0.20   
                 Vừa                            0.00               0.15   
                 Cao                            0.00               0.15   
Ridge six        Thấp                           0.04               0.31   
                 Vừa                            0.00               0.15   
                 Cao                            0.00               0.15   

                                   Số lệnh dự kiến  
Mô hình          Nhóm thanh khoản                   
LightGBM finance Thấp                        89757  
                 Vừa                         89958  
                 Cao                         90199  
Ridge six        Thấp                        87325  
                 Vừa                         87535  
                 Cao                         87758

Mã cổ phiếu
2761    140
9733    110
1981     98
5918     98
4365     95
1787     81
4628     81
4621     80
Name: Số lệnh gặp thiếu giá, dtype: int64

Tháng
2020-09    844
2021-10     91
2020-10     86
2019-02     73
2019-04     73
2019-03     69
Freq: M, Name: Số lệnh gặp thiếu giá theo tháng, dtype: int64

Phiên riêng  \
Mô hình          Số mã mỗi vế Tỷ lệ trọng số cực trị                
Ridge six        100          2.0                             700   
                 300          2.0                             700   
                 200          1.0                             700   
LightGBM finance 100          2.0                             700   
                 300          2.0                             700   
                 200          1.0                             700   
Ridge six        200          2.0                             700   
LightGBM finance 200          2.0                             700   

                                                      Phiên chung  \
Mô hình          Số mã mỗi vế Tỷ lệ trọng số cực trị                
Ridge six        100          2.0                             700   
                 300          2.0                             700   
                 200          1.0                             700   
LightGBM finance 100          2.0                             700   
                 300          2.0                             700   
                 200          1.0                             700   
Ridge six        200          2.0                             700   
LightGBM finance 200          2.0                             700   

                                                      Spread (bps/phiên)  \
Mô hình          Số mã mỗi vế Tỷ lệ trọng số cực trị                       
Ridge six        100          2.0                                11.9395   
                 300          2.0                                 8.8610   
                 200          1.0                                 9.9644   
LightGBM finance 100          2.0                                 7.4707   
                 300          2.0                                 4.9236   
                 200          1.0                                 5.6349   
Ridge six        200          2.0                                10.3911   
LightGBM finance 200          2.0                                 6.0520   

                                                      Sharpe  
Mô hình          Số mã mỗi vế Tỷ lệ trọng số cực trị          
Ridge six        100          2.0                     0.0761  
                 300          2.0                     0.0735  
                 200          1.0                     0.0762  
LightGBM finance 100          2.0                     0.1195  
                 300          2.0                     0.1176  
                 200          1.0                     0.1217  
Ridge six        200          2.0                     0.0768  
LightGBM finance 200          2.0                     0.1252

Nhóm thanh khoản dùng giá trị giao dịch quá khứ. Rổ mã và mức độ phân tán có thể khác nhau giữa các nhóm. Bảng cuối giữ cùng phiên chấm khi thay số mã hoặc hệ số trọng số và vẫn báo số phiên riêng của từng cấu hình. Hai tỷ lệ thiếu giá vào và quá hạn thoát ở trên cùng chia cho toàn bộ lệnh dự kiến của nhóm.